In [9]:
%run init_env.py

Added to Python path: D:\git\cs5305\genai-langchain
Environment initialization completed successfully.


# Multi-Agent Event Planner

In [10]:
from databricks_langchain import ChatDatabricks
from typing import Dict, Any
from tavily import TavilyClient
from langchain.tools import tool

llm = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct", temperature=0, max_tokens=500)
tavily_client = TavilyClient()

@tool
def web_search(query: str, search_number: int, max_search_number: int) -> Dict[str, Any]:
    """Search the web for information. You must track your search count by providing
    search_number (starting at 1) and max_search_number on every call.
    Queries must use only plain text characters. Do not use accented or special characters     
      (e.g., use 'capacite' instead of 'capacité').
    """
    if search_number > max_search_number:
        return {"message": "Search limit reached. Please summarize your findings and provide your final answer."}
    try:
        return tavily_client.search(query)
    except Exception as e:
        return {"error": str(e)}

In [11]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

@tool
def query_playlist_db(query: str) -> str:

    """Query the database for playlist information"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error querying database: {e}"

## Create State

In [12]:
from langchain.agents import AgentState

class EventState(AgentState):
    origin: str
    destination: str
    guest_count: str
    genre: str

## Create Subagents


In [13]:
# Venue agent
venue_agent = create_agent(
    model=llm,
    tools=[web_search],
    system_prompt="""
    You are a venue specialist. Search for venues in the desired location, and with the desired capacity.
    You are not allowed to ask any more follow up questions, you must find the best venue options based on the following criteria:
    - Price (lowest)
    - Capacity (exact match)
    - Reviews (highest)
    You may need to make multiple searches to iteratively find the best options. 
    You have a suggested limit of 5 web searches. Count every web_search call you make.
    After 5 searches, you should stop searching and summarize the best options you have
    found so far.
    """
)

In [14]:
# Playlist agent
playlist_agent = create_agent(
    model=llm,
    tools=[query_playlist_db],
    system_prompt="""
    You are a playlist specialist. Query the sql database and curate the perfect playlist for an event given a genre.
    Once you have your playlist, calculate the total duration and cost of the playlist, each song has an associated price.
    If you run into errors when querying the database, try to fix them by making changes to the query.
    Do not come back empty handed, keep trying to query the db until you find a list of songs.

    This is a SQLite database. Before writing any data queries, first discover the schema.
    """
)

## Main Coordinator


In [15]:
from langchain.tools import ToolRuntime
from langchain.messages import HumanMessage, ToolMessage
from langgraph.types import Command

@tool
def search_venues(runtime: ToolRuntime) -> str:
    """Venue agent chooses the best venue for the given location and capacity."""
    destination = runtime.state["destination"]
    capacity = runtime.state["guest_count"]
    query = f"Find event venues in {destination} for {capacity} guests"
    response = venue_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content

@tool
def suggest_playlist(runtime: ToolRuntime) -> str:
    """Playlist agent curates the perfect playlist for the given genre."""
    genre = runtime.state["genre"]
    query = f"Find {genre} tracks for event playlist"
    response = playlist_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content

@tool
def update_state(origin: str, destination: str, guest_count: str, genre: str, runtime: ToolRuntime) -> str:
    """Update the state when you know all of the values: origin, destination, guest_count, genre. 
    This tool must be called alone, without any other tool calls. It must complete and return to make,
    the information available to other tools."""
    return Command(update={
        "origin": origin, 
        "destination": destination, 
        "guest_count": guest_count, 
        "genre": genre, 
        "messages": [ToolMessage("Successfully updated state", tool_call_id=runtime.tool_call_id)]}
        )


In [16]:
from databricks_langchain import ChatDatabricks
from langchain.agents import create_agent

coordinator = create_agent(
    model=ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct", temperature=0, max_tokens=500),
    tools=[search_venues, suggest_playlist, update_state],
    state_schema=EventState,
    system_prompt="""
    You are a event planner & coordinator. 
    First find all the information you need to update the state. When you have the information, update the state.
    Once that has completed and returned, you can delegate the tasks 
    to your specialists for venues, and playlists.
    Once you have received their answers, coordinate the perfect event for me.
    """
)

## Testing the event planner ...


In [17]:
from langchain.messages import HumanMessage

response = await coordinator.ainvoke(
    {
        "messages": [HumanMessage(content="I'm from Lahore and I'd like to schedule an event in Gujranwala for 100 guests. also create a play list with jazz genre.  Make sure that playlist has multiple artists")],
    },
    config={"tags": ["WP"]},  #tag traces to make them easy to find in Langsmith.
)

msg_list = response["messages"]
print(msg_list[-1].content)

For your event in Gujranwala with 100 guests, I recommend Adam Palace as the venue. It perfectly matches your guest capacity, offers complete food service, central air conditioning, ample parking, and has positive reviews.

For the jazz playlist, I have curated a set of 20 tracks from multiple artists, totaling approximately 91 minutes and 39 seconds of music. The total cost for the playlist tracks is $19.80.

Would you like me to proceed with booking the venue Adam Palace, or would you prefer details for any other venue? Also, do you want assistance with anything else for your event?


In [18]:
def send_message(msg: str, message_history):

    response = coordinator.invoke(
        {
            "messages": message_history + [
                HumanMessage(
                    content=msg
                )
            ]
        },
        config={"tags": ["WP"]},
    )

    updated_messages = response["messages"]
    print(updated_messages[-1].content)
    return updated_messages

In [19]:
msg_list = send_message("Can you please display the playlist?", msg_list)
msg_list[-1].content

Certainly! Here is the jazz playlist curated for your event:

1. Desafinado - 185,338 ms
2. Garota De Ipanema - 285,048 ms
3. Samba De Uma Nota Só (One Note Samba) - 137,273 ms
4. Por Causa De Você - 169,900 ms
5. Ligia - 251,977 ms
6. Fotografia - 129,227 ms
7. Dindi (Dindi) - 253,178 ms
8. Se Todos Fossem Iguais A Você (Instrumental) - 134,948 ms
9. Falando De Amor - 219,663 ms
10. Angela - 169,508 ms
11. Corcovado (Quiet Nights Of Quiet Stars) - 205,662 ms
12. Outra Vez - 126,511 ms
13. O Boto (Bôto) - 366,837 ms
14. Canta, Canta Mais - 271,856 ms
15. Quadrant - 261,851 ms
16. Snoopy's search-Red baron - 456,071 ms
17. Spanish moss-"A sound portrait"-Spanish moss - 248,084 ms
18. Moon germs - 294,060 ms
19. Stratus - 582,086 ms
20. The pleasant pheasant - 318,066 ms

This playlist offers a rich variety of jazz tracks from multiple artists, perfect for setting the mood at your event. Let me know if you'd like any adjustments or additional help!


'Certainly! Here is the jazz playlist curated for your event:\n\n1. Desafinado - 185,338 ms\n2. Garota De Ipanema - 285,048 ms\n3. Samba De Uma Nota Só (One Note Samba) - 137,273 ms\n4. Por Causa De Você - 169,900 ms\n5. Ligia - 251,977 ms\n6. Fotografia - 129,227 ms\n7. Dindi (Dindi) - 253,178 ms\n8. Se Todos Fossem Iguais A Você (Instrumental) - 134,948 ms\n9. Falando De Amor - 219,663 ms\n10. Angela - 169,508 ms\n11. Corcovado (Quiet Nights Of Quiet Stars) - 205,662 ms\n12. Outra Vez - 126,511 ms\n13. O Boto (Bôto) - 366,837 ms\n14. Canta, Canta Mais - 271,856 ms\n15. Quadrant - 261,851 ms\n16. Snoopy\'s search-Red baron - 456,071 ms\n17. Spanish moss-"A sound portrait"-Spanish moss - 248,084 ms\n18. Moon germs - 294,060 ms\n19. Stratus - 582,086 ms\n20. The pleasant pheasant - 318,066 ms\n\nThis playlist offers a rich variety of jazz tracks from multiple artists, perfect for setting the mood at your event. Let me know if you\'d like any adjustments or additional help!'

In [20]:
msg_list = send_message("Please shorten the playlist by half.  Display the list with complete names and album details", msg_list)

KeyError: 'genre'

### What happend??? State is no longer visible to the LLM.

Lets try to fix the problem with explicitly retreiving/setting the state as well along with message list

In [21]:
from langchain.messages import HumanMessage

response = await coordinator.ainvoke(
    {
        "messages": [HumanMessage(content="I'm from Lahore and I'd like to schedule an event in Gujranwala for 100 guests. also create a play list with jazz genre.  Make sure that playlist has multiple artists")],
    },
    config={"tags": ["WP"], "recursion_limit": 40},  #tag traces to make them easy to find in Langsmith. Increase number of steps the agent can take to 40.
)

coordinator_state = {
    "origin": response.get("origin"),
    "destination": response.get("destination"),
    "guest_count": response.get("guest_count"),
    "genre": response.get("genre"),
}

msg_list = response["messages"]
print(msg_list[-1].content)

For your event in Gujranwala with 100 guests, I recommend the following venue options:

1. Adam Palace Gujranwala
   - Capacity: 100 to 300 guests
   - Features: Central AC and heating, prime location on GT Road
   - Price: Affordable rates
   - Positive reviews for being elegant and well-located

2. Royal Pepper Banquets (Tulip)
   - Capacity: 100 to 800 guests
   - Positive reviews for decoration, space, and food

Both venues are suitable for your guest count and have good amenities.

Regarding the jazz playlist, I have curated a list of 20 jazz tracks from multiple artists. The total duration is approximately 99.3 minutes (about 1 hour and 39 minutes), and the estimated cost for the tracks is $19.80.

Would you like me to assist with booking the venue or provide contact details? Also, do you want the playlist in a specific format or platform?


In [22]:
def send_message_v2(msg: str, message_history):
    global coordinator_state

    current_state = {
        key: value
        for key, value in globals().get("coordinator_state", {}).items()
        if value is not None
    }

    response = coordinator.invoke(
        {
            **current_state,
            "messages": message_history + [
                HumanMessage(
                    content=msg
                )
            ]
        },
        config={"tags": ["WP"], "recursion_limit": 40},
    )

    coordinator_state = {
        "origin": response.get("origin"),
        "destination": response.get("destination"),
        "guest_count": response.get("guest_count"),
        "genre": response.get("genre"),
    }

    updated_messages = response["messages"]
    print(updated_messages[-1].content)
    return updated_messages

In [23]:
msg_list = send_message_v2("Can you please display the playlist?", msg_list)
msg_list[-1].content

Certainly! Here is the jazz playlist for your event with 20 tracks from multiple artists:

1. Desafinado - 185,338 ms
2. Garota De Ipanema - 285,048 ms
3. Samba De Uma Nota Só (One Note Samba) - 137,273 ms
4. Por Causa De Você - 169,900 ms
5. Ligia - 251,977 ms
6. Fotografia - 129,227 ms
7. Dindi (Dindi) - 253,178 ms
8. Se Todos Fossem Iguais A Você (Instrumental) - 134,948 ms
9. Falando De Amor - 219,663 ms
10. Angela - 169,508 ms
11. Corcovado (Quiet Nights Of Quiet Stars) - 205,662 ms
12. Outra Vez - 126,511 ms
13. O Boto (Bôto) - 366,837 ms
14. Canta, Canta Mais - 271,856 ms
15. Quadrant - 261,851 ms
16. Snoopy's search-Red baron - 456,071 ms
17. Spanish moss-"A sound portrait"-Spanish moss - 248,084 ms
18. Moon germs - 294,060 ms
19. Stratus - 582,086 ms
20. The pleasant pheasant - 318,066 ms

If you'd like, I can also help you with the playlist file or streaming links.


'Certainly! Here is the jazz playlist for your event with 20 tracks from multiple artists:\n\n1. Desafinado - 185,338 ms\n2. Garota De Ipanema - 285,048 ms\n3. Samba De Uma Nota Só (One Note Samba) - 137,273 ms\n4. Por Causa De Você - 169,900 ms\n5. Ligia - 251,977 ms\n6. Fotografia - 129,227 ms\n7. Dindi (Dindi) - 253,178 ms\n8. Se Todos Fossem Iguais A Você (Instrumental) - 134,948 ms\n9. Falando De Amor - 219,663 ms\n10. Angela - 169,508 ms\n11. Corcovado (Quiet Nights Of Quiet Stars) - 205,662 ms\n12. Outra Vez - 126,511 ms\n13. O Boto (Bôto) - 366,837 ms\n14. Canta, Canta Mais - 271,856 ms\n15. Quadrant - 261,851 ms\n16. Snoopy\'s search-Red baron - 456,071 ms\n17. Spanish moss-"A sound portrait"-Spanish moss - 248,084 ms\n18. Moon germs - 294,060 ms\n19. Stratus - 582,086 ms\n20. The pleasant pheasant - 318,066 ms\n\nIf you\'d like, I can also help you with the playlist file or streaming links.'

In [24]:
msg_list = send_message_v2("Please shorten the playlist by half.  Display the list with complete names and album details", msg_list)

Here is a shortened jazz playlist with 10 tracks for your event:

1. Desafinado - Duration: 185,338 ms
2. Garota De Ipanema - Duration: 285,048 ms
3. Samba De Uma Nota Só (One Note Samba) - Duration: 137,273 ms
4. Por Causa De Você - Duration: 169,900 ms
5. Ligia - Duration: 251,977 ms
6. Fotografia - Duration: 129,227 ms
7. Dindi (Dindi) - Duration: 253,178 ms
8. Se Todos Fossem Iguais A Você (Instrumental) - Duration: 134,948 ms
9. Falando De Amor - Duration: 219,663 ms
10. Angela - Duration: 169,508 ms

The total duration is approximately 30 minutes and 37 seconds, and the total cost is $9.90.

Unfortunately, album details were not provided in the data I have. Would you like me to look up the album information for these tracks?


### Not the ideal way to manage agent state!  Lets fix and simply the code with proper state management!